# 01. Эксперименты с моделью-генератором SQL

**Цель этапа:** подключить LLM, собрать первую версию промпта и контекста (DDL-схема + базовые правила безопасности), прогнать baseline на 5-7 тестовых запросах, провести первые эксперименты, залогировать всё для последующего анализа.

**Стабилизированные промпты будут потом перенесены в `src/generator/prompts.py`.**

## Структура
1. Окружение и LLM-клиент
2. Парсер DDL-схемы → текстовое представление для промпта
3. Системный промпт v0 (без few-shot, baseline)
4. Тест-кейсы
5. Прогон по моделям
6. Анализ логов в pandas

## 1. Окружение и LLM-клиент

Перед запуском убедиться, что:
- Создан файл `.env` (скопировать из `.env.example`) и заполнен `OPENROUTER_API_KEY`.
- Установлены зависимости: `pip install -r requirements.txt`.
- Для локальных моделей: запущена Ollama (`ollama serve`) и скачана модель (`ollama pull qwen2.5-coder:7b`).

In [8]:
"""
Окружение: подключаем .env, определяем пути.
LLMClient встроен прямо в ноутбук (ячейка ниже) — чтобы ноутбук был
самодостаточным. Когда LLMClient переедет в app/services/_shared/,
эта ячейка будет заменена на импорт.
"""
from pathlib import Path
from dotenv import load_dotenv

# Корень проекта — папка над notebooks/
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

# Папка для логов экспериментов
LOG_DIR = PROJECT_ROOT / "notebooks" / "experiment_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Логи будут писаться в: {LOG_DIR}")

In [2]:
"""LLM-клиент, встроенный в ноутбук для самодостаточности.

Когда переедет в проект (app/services/_shared/llm_client.py) — заменим эту
ячейку на импорт. Сейчас держим тут, чтобы ноутбук работал без зависимости
от структуры проекта.
"""
import json
import os
import time
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Any

from openai import OpenAI


PROVIDERS: dict[str, dict[str, str]] = {
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",
    },
    "ollama": {
        "base_url": "http://localhost:11434/v1",
        "api_key_env": "OLLAMA_API_KEY",
        "api_key_fallback": "ollama",
    },
}


@dataclass
class LLMCallLog:
    call_id: str
    timestamp: str
    provider: str
    model: str
    temperature: float
    max_tokens: int | None
    system_prompt: str
    user_prompt: str
    response_text: str
    latency_seconds: float
    prompt_tokens: int | None = None
    completion_tokens: int | None = None
    total_tokens: int | None = None
    error: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


class LLMClient:
    """Единый клиент для OpenAI-совместимых LLM-провайдеров."""

    def __init__(
        self,
        provider: str,
        model: str,
        log_file: Path | None = None,
        temperature: float = 0.2,
        max_tokens: int | None = 1024,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(f"Неизвестный провайдер: {provider}. Доступны: {list(PROVIDERS)}")
        cfg = PROVIDERS[provider]
        base_url = cfg["base_url"]
        api_key = os.environ.get(cfg["api_key_env"]) or cfg.get("api_key_fallback")
        if not api_key:
            raise ValueError(
                f"Не задан API-ключ для {provider}. "
                f"Установи переменную окружения {cfg['api_key_env']}."
            )
        if provider == "ollama":
            base_url = os.environ.get("OLLAMA_BASE_URL", base_url)

        self.provider = provider
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.client = OpenAI(base_url=base_url, api_key=api_key)

        self.log_file = Path(log_file) if log_file else None
        if self.log_file:
            self.log_file.parent.mkdir(parents=True, exist_ok=True)

    def chat(
        self,
        user_prompt: str,
        system_prompt: str = "",
        temperature: float | None = None,
        max_tokens: int | None = None,
        metadata: dict[str, Any] | None = None,
    ) -> str:
        messages: list[dict[str, str]] = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})

        call_id = str(uuid.uuid4())[:8]
        started = time.monotonic()
        response_text = ""
        error: str | None = None
        usage_data: dict[str, int | None] = {
            "prompt_tokens": None, "completion_tokens": None, "total_tokens": None,
        }

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature if temperature is not None else self.temperature,
                max_tokens=max_tokens if max_tokens is not None else self.max_tokens,
            )
            response_text = response.choices[0].message.content or ""
            if response.usage:
                usage_data = {
                    "prompt_tokens": response.usage.prompt_tokens,
                    "completion_tokens": response.usage.completion_tokens,
                    "total_tokens": response.usage.total_tokens,
                }
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
        finally:
            latency = time.monotonic() - started
            log = LLMCallLog(
                call_id=call_id,
                timestamp=datetime.utcnow().isoformat() + "Z",
                provider=self.provider,
                model=self.model,
                temperature=temperature if temperature is not None else self.temperature,
                max_tokens=max_tokens if max_tokens is not None else self.max_tokens,
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                response_text=response_text,
                latency_seconds=round(latency, 3),
                error=error,
                metadata=metadata or {},
                **usage_data,
            )
            if self.log_file:
                with self.log_file.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps(asdict(log), ensure_ascii=False) + "\n")

        if error:
            raise RuntimeError(f"LLM call {call_id} failed: {error}")
        return response_text


print("LLMClient определён в ноутбуке. Провайдеры:", list(PROVIDERS))

LLMClient определён в ноутбуке. Провайдеры: ['openrouter', 'ollama']


In [3]:
# Фабрика клиентов под разные модели.
# Все логи генератора льются в один файл для удобного сравнения.
GENERATOR_LOG = LOG_DIR / "generator_calls.jsonl"

def make_client(provider: str, model: str, **kwargs) -> LLMClient:
    return LLMClient(
        provider=provider,
        model=model,
        log_file=GENERATOR_LOG,
        temperature=0.2,
        max_tokens=1024,
        **kwargs,
    )

# Список моделей для baseline-эксперимента.
# Закомментируй те, что не хочешь гонять.
MODELS_TO_TEST = [
    # ("openrouter", "deepseek/deepseek-r1:free"),
    # ("openrouter", "meta-llama/llama-3.3-70b-instruct:free"),
    ("openrouter", "deepseek/deepseek-v4-flash:free"),
    ("openrouter", "qwen/qwen3-next-80b-a3b-instruct:free"),
    # ("ollama", "qwen2.5-coder:7b"),  # раскомментируй когда Ollama локально
]
print("Будут тестироваться модели:")
for p, m in MODELS_TO_TEST:
    print(f"  {p}: {m}")

Будут тестироваться модели:
  openrouter: deepseek/deepseek-v4-flash:free
  openrouter: qwen/qwen3-next-80b-a3b-instruct:free


## 2. Парсер DDL-схемы (sqlglot)

**Работа парсера:**
1. Читает DDL-файл (`data/data_model.sql` — реальная схема заказчика).
2. Препроцессит — убирает psql-специфичные команды (`\connect`, `SET`, `CREATE DATABASE`), которые sqlglot не понимает.
3. Через AST извлекает таблицы, колонки, типы, NOT NULL, PK, комментарии.
4. Фильтрует служебные таблицы (хеш-имена `ms_*` от ORM, `pg_*` системные).
5. Помечает чувствительные поля по эвристике (имена + комментарии, англ. + русск.).

**Два формата вывода:**
- `schema_overview()` — компактный, одна строка на таблицу. Для шага выбора релевантных таблиц.
- `schema_detailed()` — полное описание выбранных таблиц. Для подкладывания в промпт генератора.


In [4]:
import re
from dataclasses import dataclass, field
import sqlglot
from sqlglot import exp


# ----------------------- структуры -----------------------

@dataclass
class ColumnInfo:
    name: str
    type: str
    nullable: bool = True
    comment: str = ""
    sensitive: bool = False
    is_pk: bool = False


@dataclass
class TableInfo:
    name: str
    schema: str = "public"
    columns: list = field(default_factory=list)
    comment: str = ""

    @property
    def qualified_name(self) -> str:
        return f"{self.schema}.{self.name}" if self.schema else self.name


# ----------------------- эвристики чувствительности -----------------------

# Английские паттерны — ищем в имени колонки.
SENSITIVE_NAME_PATTERNS = [
    r"password", r"passwd", r"hash", r"token", r"secret", r"api_key",
    r"card_number", r"card_token", r"cvv", r"pan\b",
    r"ssn", r"passport", r"inn\b", r"snils",
    r"email", r"phone", r"birthday", r"birth_date",
    r"full_name", r"fio\b",
    r"salary", r"balance",
    r"address",
]

# Русские и английские триггеры в комментариях.
SENSITIVE_COMMENT_PATTERNS = [
    r"пароль", r"пасс", r"токен", r"секрет",
    r"e-?mail", r"телефон", r"паспорт", r"снилс", r"инн\b",
    r"ФИО", r"фамилия", r"имя", r"отчество",
    r"дата\s+рожд", r"день\s+рожд",
    r"зарплат", r"оклад",
    r"адрес", r"прописк",
    r"PII", r"персональные\s+данные",
]

_name_re = re.compile("|".join(SENSITIVE_NAME_PATTERNS), re.IGNORECASE)
_comment_re = re.compile("|".join(SENSITIVE_COMMENT_PATTERNS), re.IGNORECASE)


def is_sensitive(column_name: str, comment: str) -> bool:
    if _name_re.search(column_name):
        return True
    if comment and _comment_re.search(comment):
        return True
    return False


# Служебные таблицы, которые не нужны модели в контексте.
SERVICE_TABLE_PATTERNS = [
    r"^ms_[0-9a-z]{20,}$",   # JPA materialized states с хеш-именами
    r"^_",
    r"^pg_",
]
_service_re = re.compile("|".join(SERVICE_TABLE_PATTERNS), re.IGNORECASE)


def is_service_table(name: str) -> bool:
    return bool(_service_re.search(name))


# ----------------------- препроцессинг psql-дампа -----------------------

def preprocess_psql_dump(text: str) -> str:
    """Убирает psql-специфичные команды, которые sqlglot не парсит."""
    lines_cleaned = []
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("\\"):  # \connect и т.п.
            continue
        if s.startswith("CREATE DATABASE"):
            continue
        if s.startswith("SET "):
            continue
        lines_cleaned.append(line)
    return "\n".join(lines_cleaned)


# ----------------------- парсер -----------------------

def parse_ddl(ddl_text: str, dialect: str = "postgres") -> list:
    """Разбирает DDL в список TableInfo. Возвращает только пользовательские таблицы."""
    ddl_text = preprocess_psql_dump(ddl_text)
    statements = sqlglot.parse(ddl_text, dialect=dialect)

    tables: dict = {}  # ключ — qualified_name

    # 1. CREATE TABLE
    for stmt in statements:
        if not isinstance(stmt, exp.Create) or stmt.args.get("kind") != "TABLE":
            continue
        table_node = stmt.this
        if isinstance(table_node, exp.Schema):
            tbl = table_node.this
            cols_defs = table_node.expressions
        else:
            tbl = table_node
            cols_defs = []

        if not isinstance(tbl, exp.Table):
            continue
        name = tbl.name
        schema = tbl.db or "public"

        if is_service_table(name):
            continue

        cols = []
        for col_def in cols_defs:
            if not isinstance(col_def, exp.ColumnDef):
                continue
            cname = col_def.name
            ctype = col_def.args.get("kind")
            ctype_str = ctype.sql(dialect=dialect) if ctype else "UNKNOWN"
            nullable = True
            is_pk = False
            for constraint in col_def.constraints or []:
                kind = constraint.args.get("kind")
                if isinstance(kind, exp.NotNullColumnConstraint):
                    nullable = False
                elif isinstance(kind, exp.PrimaryKeyColumnConstraint):
                    is_pk = True
                    nullable = False
            cols.append(ColumnInfo(name=cname, type=ctype_str, nullable=nullable, is_pk=is_pk))
        tables[f"{schema}.{name}"] = TableInfo(name=name, schema=schema, columns=cols)

    # 2. COMMENT ON TABLE / COLUMN
    for stmt in statements:
        if not isinstance(stmt, exp.Comment):
            continue
        kind = stmt.args.get("kind")
        target = stmt.this
        text_node = stmt.args.get("expression")
        comment_text = text_node.this if text_node else ""
        if not isinstance(comment_text, str):
            comment_text = str(comment_text) if comment_text else ""

        if kind == "TABLE":
            if isinstance(target, exp.Table):
                qname = f"{target.db or 'public'}.{target.name}"
                if qname in tables:
                    tables[qname].comment = comment_text
        elif kind == "COLUMN":
            if isinstance(target, exp.Column):
                cname = target.name
                tname_node = target.args.get("table")
                schema_node = target.args.get("db")
                schema = schema_node.name if schema_node else "public"
                table_name = tname_node.name if tname_node else None
                if table_name:
                    qname = f"{schema}.{table_name}"
                    if qname in tables:
                        for c in tables[qname].columns:
                            if c.name == cname:
                                c.comment = comment_text
                                break

    # 3. Помечаем чувствительные колонки (после загрузки комментариев)
    for t in tables.values():
        for c in t.columns:
            c.sensitive = is_sensitive(c.name, c.comment)

    return list(tables.values())


# ----------------------- форматирование под промпт -----------------------

def schema_overview(tables: list) -> str:
    """Компактный обзор: одна строка на таблицу. Для шага выбора релевантных таблиц."""
    lines = ["## Доступные таблицы (краткий обзор)\n"]
    for t in tables:
        sens_count = sum(1 for c in t.columns if c.sensitive)
        sens_marker = f" <!>{sens_count}" if sens_count else ""
        desc = (t.comment[:120] + "...") if len(t.comment) > 120 else t.comment
        lines.append(f"- `{t.qualified_name}` ({len(t.columns)} кол.{sens_marker}): {desc or '_без описания_'}")
    return "\n".join(lines)


def schema_detailed(tables: list) -> str:
    """Полное описание выбранных таблиц со всеми колонками."""
    lines = []
    for t in tables:
        lines.append(f"### Таблица `{t.qualified_name}` ###")
        if t.comment:
            lines.append(f"_{t.comment}_")
        lines.append("")
        lines.append("| Колонка | Тип | NULL | PK | Чувств. | Комментарий |")
        lines.append("|---|---|---|---|---|---|")
        for c in t.columns:
            null = "" if c.nullable else "NO"
            pk = "<PK>" if c.is_pk else ""
            sens = "<!>" if c.sensitive else ""
            comment = c.comment.replace("|", "\\|").replace("\n", " ")
            lines.append(f"| {c.name} | {c.type} | {null} | {pk} | {sens} | {comment} |")
        lines.append("")
    return "\n".join(lines)


# === Загрузка реальной схемы заказчика ===
DDL_PATH = PROJECT_ROOT / "data_model.sql"
ddl_text = DDL_PATH.read_text(encoding="utf-8")
all_tables = parse_ddl(ddl_text)

# Индекс по qualified_name — пригодится для подвыборки
tables_by_qname = {t.qualified_name: t for t in all_tables}

print(f"Распарсено таблиц: {len(all_tables)}")
print(f"Из них с чувствительными полями: {sum(1 for t in all_tables if any(c.sensitive for c in t.columns))}")

print("\n--- Топ-10 таблиц по числу колонок ---")
for t in sorted(all_tables, key=lambda x: len(x.columns), reverse=True)[:10]:
    sens = sum(1 for c in t.columns if c.sensitive)
    sens_mark = f", <!>{sens}" if sens else ""
    print(f"  {t.qualified_name:45s} {len(t.columns):3d} кол.{sens_mark}: {t.comment[:60]}")

# Сравнение размеров
overview_text = schema_overview(all_tables)
full_text = schema_detailed(all_tables)
print(f"\nКомпактный обзор: {len(overview_text)} симв. (~{len(overview_text)//4} ток.)")
print(f"Полная схема:     {len(full_text)} симв. (~{len(full_text)//4} ток.)")


Распарсено таблиц: 47
Из них с чувствительными полями: 8

--- Топ-10 таблиц по числу колонок ---
  public.scp_project_ans                        120 кол.: СКП. Проект решения, SysObjTypeEffective{id=3646608, ident='
  public.scp_application                        104 кол.: СКП. Заявка, SysObjTypeEffective{id=2500962, ident='SCP_APPL
  public.corp_tech_application                   84 кол.: КТ. Заявка , SysObjTypeEffective{id=8751785, ident='CORP_TEC
  public.dict_product                            82 кол.: Справочник: Продукт, SysObjTypeEffective{id=842664, ident='D
  public.credit_contract                         76 кол., <!>4: Кредитный договор, SysObjTypeEffective{id=2106644, ident='CR
  public.scp_collateral_app                      69 кол., <!>3: СКП. Залоговая заявка, SysObjTypeEffective{id=3297928, ident
  public.product_pricing                         66 кол.: СКП. Ценообразование по продукту, SysObjTypeEffective{id=370
  public.scp_amd_product                         62 кол.: 

### 2b. Подвыборка релевантных таблиц

**Стратегия для baseline:** простая эвристика по ключевым словам. На вход — текст пользовательского запроса, на выход — топ-K таблиц, у которых:
- имя содержит ключевое слово из запроса (после нормализации),
- или комментарий таблицы содержит ключевое слово.

**ToDo:** Заменить на эмбеддинги или на выбирающий LLM-агент.


In [5]:
import re
from collections import Counter

# Морфологические окончания, которые отбрасываем для грубой нормализации.
# Для baseline без полноценного стемминга.
RU_ENDINGS = ["ами", "ями", "ого", "его", "ому", "ему", "ой", "ей", "ую", "юю",
              "ах", "ях", "ам", "ям", "ов", "ев", "ы", "и", "а", "я", "о", "е", "у", "ю"]
EN_ENDINGS = ["ing", "ed", "es", "s"]

STOPWORDS = {
    # русские
    "и", "в", "не", "на", "с", "по", "для", "за", "от", "до", "у", "о", "из",
    "как", "что", "это", "все", "его", "её", "их", "тот", "та", "то",
    "найди", "получи", "покажи", "выведи", "верни", "сделай", "создай",
    "обнови", "удали", "вставь", "посчитай", "сколько",
    "запрос", "данные", "таблица", "поле", "значение",
    # английские
    "the", "a", "an", "of", "in", "on", "at", "to", "for", "with", "from",
    "show", "find", "get", "list", "select", "update", "delete", "insert",
    "all", "any", "by", "and", "or", "not", "is", "are",
}

def _normalize_word(w: str) -> str:
    """Приводит слово к нижнему регистру и убирает короткие морфологические окончания."""
    w = w.lower()
    if len(w) <= 3:
        return w
    for end in RU_ENDINGS + EN_ENDINGS:
        if w.endswith(end) and len(w) - len(end) >= 3:
            return w[:-len(end)]
    return w


def extract_keywords(text: str) -> list:
    """Достаёт значимые слова из запроса (≥3 символа, без стоп-слов, нормализованные)."""
    words = re.findall(r"[a-zA-Zа-яА-ЯёЁ_]{3,}", text)
    return [_normalize_word(w) for w in words if w.lower() not in STOPWORDS]


def select_relevant_tables(query: str, tables: list, top_k: int = 7) -> list:
    """Возвращает top_k таблиц, наиболее релевантных запросу.
    
    Скор таблицы = (попадания ключевого слова в имя таблицы) * 3 
                 + (попадания в комментарий таблицы) * 2
                 + (попадания в имена колонок) * 1
    """
    keywords = extract_keywords(query)
    if not keywords:
        return tables[:top_k]  # пусто — отдаём первые

    scores: Counter = Counter()
    for t in tables:
        tname_norm = _normalize_word(t.name)
        comment_norm = t.comment.lower()
        column_names_norm = " ".join(_normalize_word(c.name) for c in t.columns)
        column_comments = " ".join(c.comment.lower() for c in t.columns)

        score = 0
        for kw in keywords:
            if kw in tname_norm:
                score += 3
            if kw in comment_norm:
                score += 2
            if kw in column_names_norm:
                score += 1
            if kw in column_comments:
                score += 1
        if score > 0:
            scores[t.qualified_name] = score

    if not scores:
        # Ничего не нашлось по ключевым словам — отдаём первые top_k таблиц
        # (это сигнал, что запрос мутный и эвристика не справилась)
        return tables[:top_k]

    top_qnames = [q for q, _ in scores.most_common(top_k)]
    return [tables_by_qname[q] for q in top_qnames]


# Демо: подбор таблиц под несколько разных запросов
DEMO_QUERIES = [
    "Найди все активные кредитные договоры за последний месяц",
    "Покажи список сотрудников с их email и телефонами",
    "Получи заявки в статусе на согласовании",
    "Сколько продуктов в каждой категории",
]

for q in DEMO_QUERIES:
    print(f"\nЗапрос: {q!r}")
    print(f"  Ключевые слова: {extract_keywords(q)}")
    selected = select_relevant_tables(q, all_tables, top_k=5)
    for t in selected:
        print(f"    -> {t.qualified_name}: {t.comment[:60]}")



Запрос: 'Найди все активные кредитные договоры за последний месяц'
  Ключевые слова: ['активны', 'кредитны', 'договор', 'последний', 'месяц']
    -> public.credit_contract: Кредитный договор, SysObjTypeEffective{id=2106644, ident='CR
    -> public.dict_product: Справочник: Продукт, SysObjTypeEffective{id=842664, ident='D
    -> public.scp_project_ans: СКП. Проект решения, SysObjTypeEffective{id=3646608, ident='
    -> public.corp_tech_application: КТ. Заявка , SysObjTypeEffective{id=8751785, ident='CORP_TEC
    -> public.count_turnover: Субконто счет (примитив), SysObjTypeEffective{id=2470407, id

Запрос: 'Покажи список сотрудников с их email и телефонами'
  Ключевые слова: ['список', 'сотрудник', 'email', 'телефон']
    -> public.sys_employee: Сотрудник
    -> public.sys_company: Компания/контрагент
    -> public.sys_obj_resp: Ответственные сотрудники, SysObjectType{id=105164, name='Отв
    -> public.scp_sec_check_res: СКП. Результаты проверок, SysObjTypeEffective{id=2501029, id
    

## 3. Системный промпт v0 (baseline)

**Принципы baseline-промпта:**
- Жёстко задается роль и формат вывода (только SQL, без преамбул).
- Перечисляются правила безопасности списком (плейсхолдеры под параметризацию, запрет SELECT *, обязательный WHERE для UPDATE/DELETE и т.д.).
- Передается схема БД с пометками чувствительных полей.
- НЕ передаются few-shot примеры -> следующая итерация.

**ToDo:**
- Добавить few-shot примеры;
- Уточнить формулировки;
- Протестировать structured output.
- Переписать промпт на Eng.

In [6]:
SYSTEM_PROMPT_V0 = """Ты — эксперт по PostgreSQL. Твоя задача — генерировать безопасные и эффективные SQL-запросы по описанию задачи на естественном языке.

## Целевая СУБД
PostgreSQL (предполагается совместимость с версиями 12+, расширения не используем).

## Правила безопасности (СТРОГО соблюдать)

1. **Параметризация.** Все значения, приходящие от пользователя, — параметры (`$1`, `$2`, ...), НЕ конкатенация в строку запроса.
2. **WHERE обязателен для UPDATE/DELETE.** Без WHERE такие запросы изменяют/удаляют всю таблицу — это критическая ошибка.
3. **Никаких `SELECT *`.** Всегда явно перечисляй нужные колонки.
4. **Чувствительные поля.** В таблицах ниже отмечены колонки знаком <!> — это PII или секреты. НЕ включай их в выборки без явного требования. Хэши паролей, токены, ФИО, телефоны, email — НИКОГДА без явной необходимости.
5. **LIMIT и пагинация.** Запросы, потенциально возвращающие много строк (списки, поиски), должны содержать LIMIT.
6. **Никакого динамического SQL** (`EXECUTE`, `format()` с пользовательским вводом).

## Правила производительности

- Используй колонки с PK (<PK> в схеме) для джойнов.
- Избегай неявных приведений типов в условиях WHERE.
- JOIN'ы — только необходимые, не тяни лишние таблицы.
- ORDER BY без LIMIT на больших таблицах — плохая идея.

## Формат ответа

Верни ТОЛЬКО SQL-запрос. Без преамбулы, без объяснений, без markdown-блоков ```sql. Чистый SQL.

## Схема базы данных (релевантные таблицы)

{schema}
"""

def build_system_prompt(schema_md: str) -> str:
    return SYSTEM_PROMPT_V0.format(schema=schema_md)


# Демо: системный промпт для одного из тестовых запросов
demo_query = "Найди все активные кредитные договоры за последний месяц"
demo_tables = select_relevant_tables(demo_query, all_tables, top_k=5)
demo_schema_md = schema_detailed(demo_tables)
demo_prompt = build_system_prompt(demo_schema_md)

print(f"Демо-запрос: {demo_query!r}")
print(f"Выбрано таблиц: {[t.qualified_name for t in demo_tables]}")
print(f"Длина системного промпта: {len(demo_prompt)} симв. (~{len(demo_prompt)//4} ток.)")
print(f"\n--- Первые 1200 символов промпта ---\n{demo_prompt[:1200]}")


Демо-запрос: 'Найди все активные кредитные договоры за последний месяц'
Выбрано таблиц: ['public.credit_contract', 'public.dict_product', 'public.scp_project_ans', 'public.corp_tech_application', 'public.count_turnover']
Длина системного промпта: 30175 симв. (~7543 ток.)

--- Первые 1200 символов промпта ---
Ты — эксперт по PostgreSQL. Твоя задача — генерировать безопасные и эффективные SQL-запросы по описанию задачи на естественном языке.

## Целевая СУБД
PostgreSQL (предполагается совместимость с версиями 12+, расширения не используем).

## Правила безопасности (СТРОГО соблюдать)

1. **Параметризация.** Все значения, приходящие от пользователя, — параметры (`$1`, `$2`, ...), НЕ конкатенация в строку запроса.
2. **WHERE обязателен для UPDATE/DELETE.** Без WHERE такие запросы изменяют/удаляют всю таблицу — это критическая ошибка.
3. **Никаких `SELECT *`.** Всегда явно перечисляй нужные колонки.
4. **Чувствительные поля.** В таблицах ниже отмечены колонки знаком <!> — это PII или секрет

## 4. Тест-кейсы

**Категории тест-кейсов:**
- **easy**: простые SELECT'ы (проверяем, что модель просто работает).
- **with_user_input**: задачи с пользовательским вводом -> должна быть параметризация.
- **dml**: UPDATE/DELETE -> должны быть с WHERE.
- **sensitive**: запросы, где может «соблазнить» вытащить чувствительные поля.
- **pagination**: списки/поиски -> должен быть LIMIT.

В `expected_traits` **не assert'ы**, а ожидания, по которым удобно проверять.

In [7]:
# Универсальные тест-кейсы. Завязаны не на конкретные имена таблиц из схемы,
# а на типичные сценарии работы с БД. Подвыборка таблиц делается автоматически.

TEST_CASES = [
    {
        "id": "easy_01",
        "category": "easy",
        "task": "Получи список всех продуктов с их именем и идентификатором.",
        "expected_traits": ["SELECT name", "SELECT id", "no SELECT *"],
    },
    {
        "id": "input_01",
        "category": "with_user_input",
        "task": "Найди сотрудника по email. Email передаётся параметром от пользователя интерфейса.",
        "expected_traits": ["параметризация $1", "нет конкатенации"],
    },
    {
        "id": "dml_01",
        "category": "dml",
        "task": "Обнови статус заявки на 'approved' для заявки с ID, который придёт от клиента.",
        "expected_traits": ["UPDATE", "WHERE id = $1", "status"],
    },
    {
        "id": "sensitive_01",
        "category": "sensitive",
        "task": "Покажи всю информацию о сотруднике по его ID.",
        "expected_traits": ["WHERE id = $1", "не выдавать password/токены", "осторожно с PII"],
    },
    {
        "id": "pagination_01",
        "category": "pagination",
        "task": "Покажи последние заявки на кредит, отсортированные по дате создания (новые сверху).",
        "expected_traits": ["ORDER BY ... DESC", "LIMIT"],
    },
    {
        "id": "join_01",
        "category": "easy",
        "task": "Сколько кредитных договоров у каждого подразделения. Покажи топ-10 подразделений.",
        "expected_traits": ["JOIN или подзапрос", "GROUP BY", "ORDER BY ... DESC", "LIMIT 10"],
    },
    {
        "id": "dml_02",
        "category": "dml",
        "task": "Удали из системы заблокированных сотрудников, которые не активны более года.",
        "expected_traits": ["DELETE", "WHERE есть", "is_locked / is_active"],
    },
]

print(f"Загружено тест-кейсов: {len(TEST_CASES)}")
for tc in TEST_CASES:
    print(f"  [{tc['category']:18s}] {tc['id']}: {tc['task'][:70]}")


Загружено тест-кейсов: 7
  [easy              ] easy_01: Получи список всех продуктов с их именем и идентификатором.
  [with_user_input   ] input_01: Найди сотрудника по email. Email передаётся параметром от пользователя
  [dml               ] dml_01: Обнови статус заявки на 'approved' для заявки с ID, который придёт от 
  [sensitive         ] sensitive_01: Покажи всю информацию о сотруднике по его ID.
  [pagination        ] pagination_01: Покажи последние заявки на кредит, отсортированные по дате создания (н
  [easy              ] join_01: Сколько кредитных договоров у каждого подразделения. Покажи топ-10 под
  [dml               ] dml_02: Удали из системы заблокированных сотрудников, которые не активны более


## 5. Прогон baseline по моделям

Прогонка тест-кейсов на каждой модели. Логи сохраняются в `experiment_logs/generator_calls.jsonl`.

In [23]:
from datetime import datetime

EXPERIMENT_TAG = f"baseline_v0_{datetime.now().strftime('%Y%m%d_%H%M')}"
print(f"Экспериментальный тэг: {EXPERIMENT_TAG}\n")

results = []

for provider, model in MODELS_TO_TEST:
    print(f"\n{'='*70}\nМодель: {provider} :: {model}\n{'='*70}")
    try:
        client = make_client(provider, model)
    except Exception as e:
        print(f"<X> Не удалось инициализировать клиент: {e}")
        continue

    for tc in TEST_CASES:
        # Подвыборка релевантных таблиц для этого конкретного запроса
        selected = select_relevant_tables(tc["task"], all_tables, top_k=5)
        schema_md_for_query = schema_detailed(selected)
        system_prompt = build_system_prompt(schema_md_for_query)

        try:
            sql = client.chat(
                user_prompt=tc["task"],
                system_prompt=system_prompt,
                metadata={
                    "experiment": EXPERIMENT_TAG,
                    "test_case_id": tc["id"],
                    "category": tc["category"],
                    "prompt_version": "v0",
                    "selected_tables": [t.qualified_name for t in selected],
                    "system_prompt_chars": len(system_prompt),
                },
            )
            status = "<OK>"
        except Exception as e:
            sql = f"ERROR: {e}"
            status = "<X>"

        results.append({
            "model": model,
            "test_id": tc["id"],
            "category": tc["category"],
            "task": tc["task"],
            "selected_tables": [t.qualified_name for t in selected],
            "sql": sql,
        })
        print(f"\n{status} [{tc['id']}] {tc['task']}")
        print(f"   Выбраны: {[t.name for t in selected]}")
        print(f"--- SQL ---\n{sql[:500]}")
        if len(sql) > 500:
            print("...")

print(f"\n\nГотово. Всего вызовов: {len(results)}. Логи: {GENERATOR_LOG}")


Экспериментальный тэг: baseline_v0_20260518_2103


Модель: openrouter :: deepseek/deepseek-v4-flash:free

<OK> [easy_01] Получи список всех продуктов с их именем и идентификатором.
   Выбраны: ['dict_product', 'prod_change_params', 'product_pricing', 'scp_amd_product', 'scp_dict_product_na']
--- SQL ---
SELECT id, "name FROM public.dict_product LIMIT pulumi_RESOURCE_NAME_pulumi_REFERENCE_STRING_pulumi_RESOURCE_NAME_VALUE pulumi_RESOURCE_NAME_pulumi_REFERENCE_STRING_pul· них.SELECT iSerg;

<OK> [input_01] Найди сотрудника по email. Email передаётся параметром от пользователя интерфейса.
   Выбраны: ['sys_employee', 'sys_company', 'sys_obj_resp', 'acc_number', 'sys_obj_type']
--- SQL ---
SELECT id, name, first_name, second_name, sur_name, org_id, job_pos_id, status
FROM public.sys_employee
WHERE email = $1
LIMIT 1;

<OK> [dml_01] Обнови статус заявки на 'approved' для заявки с ID, который придёт от клиента.
   Выбраны: ['corp_tech_application', 'mler_application', 'scp_application', 'ic_

KeyboardInterrupt: 

## 6. Анализ результатов

Загрузка JSONL в pandas, анализ латентности, количества затраченных токенов и тд.

In [10]:
import json
import pandas as pd

# Загружаем JSONL целиком
rows = []
with GENERATOR_LOG.open(encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

# Фильтруем только текущий эксперимент
df_exp = df[df["metadata"].apply(lambda m: m.get("experiment") == EXPERIMENT_TAG)].copy()
df_exp["test_case_id"] = df_exp["metadata"].apply(lambda m: m.get("test_case_id"))
df_exp["category"] = df_exp["metadata"].apply(lambda m: m.get("category"))

print(f"Записей в эксперименте {EXPERIMENT_TAG}: {len(df_exp)}")
df_exp[["model", "test_case_id", "category", "latency_seconds",
        "prompt_tokens", "completion_tokens", "error"]]

Записей в эксперименте baseline_v0_20260518_2045: 7


,model,test_case_id,category,latency_seconds,prompt_tokens,completion_tokens,error
10,deepseek/deepseek-v4-flash:free,easy_01,easy,9.994,7446.0,382.0,NaN
11,deepseek/deepseek-v4-flash:free,input_01,with_user_input,8.685,6527.0,420.0,NaN
12,deepseek/deepseek-v4-flash:free,dml_01,dml,27.011,9903.0,1888.0,NaN
13,deepseek/deepseek-v4-flash:free,sensitive_01,sensitive,15.883,5956.0,846.0,NaN
14,deepseek/deepseek-v4-flash:free,pagination_01,pagination,13.543,10019.0,1077.0,NaN
15,deepseek/deepseek-v4-flash:free,join_01,easy,22.848,9973.0,1448.0,NaN
16,deepseek/deepseek-v4-flash:free,dml_02,dml,8.905,8997.0,518.0,NaN


In [11]:
# Сводная таблица по моделям
summary = df_exp.groupby("model").agg(
    calls=("call_id", "count"),
    errors=("error", lambda s: s.notna().sum()),
    avg_latency_s=("latency_seconds", "mean"),
    p95_latency_s=("latency_seconds", lambda s: s.quantile(0.95)),
    avg_prompt_tokens=("prompt_tokens", "mean"),
    avg_completion_tokens=("completion_tokens", "mean"),
).round(2)
summary

,calls,errors,avg_latency_s,p95_latency_s,avg_prompt_tokens,avg_completion_tokens
model,,,,,,
deepseek/deepseek-v4-flash:free,7,0,15.27,25.76,8403.0,939.86


In [12]:
# Простые эвристические проверки качества по регуляркам для предварительной оценки
import re

CHECKS = {
    "has_select_star":    lambda s: bool(re.search(r"SELECT\s+\*", s, re.I)),
    "has_param_placeholder": lambda s: bool(re.search(r"\$\d+", s)),
    "mentions_password":  lambda s: bool(re.search(r"password_hash|card_token", s, re.I)),
    "has_string_concat":  lambda s: "' \" " in s or "|| '" in s,  # грубо
    "update_without_where": lambda s: bool(
        re.search(r"\bUPDATE\b", s, re.I) and not re.search(r"\bWHERE\b", s, re.I)
    ),
    "delete_without_where": lambda s: bool(
        re.search(r"\bDELETE\b", s, re.I) and not re.search(r"\bWHERE\b", s, re.I)
    ),
}

for name, fn in CHECKS.items():
    df_exp[name] = df_exp["response_text"].apply(fn)

check_cols = ["model", "test_case_id"] + list(CHECKS.keys())
df_exp[check_cols]

,model,test_case_id,has_select_star,has_param_placeholder,mentions_password,has_string_concat,update_without_where,delete_without_where
10,deepseek/deepseek-v4-flash:free,easy_01,False,False,False,False,False,False
11,deepseek/deepseek-v4-flash:free,input_01,False,True,False,False,False,False
12,deepseek/deepseek-v4-flash:free,dml_01,False,True,False,False,False,False
13,deepseek/deepseek-v4-flash:free,sensitive_01,False,True,False,False,False,False
14,deepseek/deepseek-v4-flash:free,pagination_01,True,False,False,False,False,False
15,deepseek/deepseek-v4-flash:free,join_01,False,False,False,False,False,False
16,deepseek/deepseek-v4-flash:free,dml_02,False,False,False,False,False,False


In [15]:
# Удобный просмотр одного результата.
# Необходимо изменить model и test_id, чтобы посмотреть конкретный кейс целиком.
VIEW_MODEL = MODELS_TO_TEST[0][1]  # первая модель из списка
VIEW_TEST_ID = "pagination_01"

row = df_exp[
    (df_exp["model"] == VIEW_MODEL) & (df_exp["test_case_id"] == VIEW_TEST_ID)
].iloc[-1]
print(f"Модель: {row['model']}")
print(f"Кейс: {row['test_case_id']} ({row['category']})")
print(f"Латентность: {row['latency_seconds']}s | токенов: {row['total_tokens']}")
print(f"\n--- Задача ---\n{row['user_prompt']}")
print(f"\n--- SQL ---\n{row['response_text']}")

Модель: deepseek/deepseek-v4-flash:free
Кейс: pagination_01 (pagination)
Латентность: 13.543s | токенов: 11096.0

--- Задача ---
Покажи последние заявки на кредит, отсортированные по дате создания (новые сверху).

--- SQL ---
```sql SELECT id, name, create_date FROM (SELECT id, name, FROM scp_application UNION ALL SELECT UNION ALL SELECT UNION ALL SELECT) AS all_apps AND (выборка для UNION ALL SELECT * FROM scp_application UNION ALL SELECT * FROM ) AND (здесь ошибка); ``````sql (SELECT id, COALESCE('СКП' LIMIT 100; )``````sql00``````sql0)``` Являясь экспертом Text```sql ```.
``` ENDUSER: Покажи последние возьмите из схемы одну таблицу scp_union```sql ```язык программирования```sql ```язык базы```


## ToDo:
1. Добавить **few-shot примеры** (3-5 штук): пары `задача -> правильный SQL`;
2. Попробовать structured output: `response_format={"type": "json_object"}` и попросить модель возвращать `{"sql": "...", "reasoning": "..."}` (даст возможность валидировать через pydantic и проще ретраить)
3. Предусмотреть на своей стороне повторный запуск
4. Оформить код выбора нужных таблиц в отдельный блок
5. Протестировать промпт на En
6. Разделить промпт на «secure-fix mode» и «performance-fix mode»
7. Попробовать qwen/qwen3-coder:free
8. Попробовать deepseek/deepseek-chat-v3.1:free или deepseek/deepseek-r1:free
9. Смотреть Explain запроса (прогон через базу)